# 013 - QAOA Version 0.02

#### Binary portfolio selection problem using the Quantum Approximate Optimization Algorithm (QAOA) based on a community recommendation.  

### Admin

In [1]:
# Data Handling 
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import datetime

# Stock element fetching 
import Quantum_Optimization_Functions as qof

# compute environment 
from tabulate import tabulate
import environment as envir

# helper functions 
from qubo_functions import (
    compute_expected_return_vector,
    compute_covariance_matrix,
    build_qubo_matrix_mean_variance,
)

# Qiskit packages 
from qiskit.circuit.library import TwoLocal
from qiskit.result import QuasiDistribution
from qiskit_aer.primitives import Sampler
from qiskit_algorithms import NumPyMinimumEigensolver, QAOA, SamplingVQE
from qiskit_algorithms.optimizers import COBYLA
from qiskit_finance.applications.optimization import PortfolioOptimization
from qiskit_finance.data_providers import RandomDataProvider
from qiskit_optimization.algorithms import MinimumEigenOptimizer



In [2]:
data, headers = envir.environment_state()
print(tabulate(data, headers=headers))

System/Library           Version
-----------------------  ---------
Python                   3.10.9
numpy                    2.2.6
torch                    2.9.1
matplotlib               3.10.8
seaborn                  0.13.2
qiskit                   1.4.4
qiskit_machine_learning  0.8.4
qiskit_optimization      0.7.0
qiskit_algorithms        0.4.0
yfinance                 1.2.0
qiskit_aer               0.17.2


### Notebook Parameters 

In [3]:
Target_Tickers = [
    "AAPL",
    "IBM",
    "JNJ",
    "CVS"
]

START_DATE = "2020-01-01"
END_DATE = "2025-01-01"

RISK_AVERSION = 0.5

print_flag = True 

### Fetch the Data 

In [4]:
# if you want just a random sample, omit the companies_list and provide a sample size 
# summary_df, Asset_Returns_DataFrame = qof.get_assets(sample_size = 5)

# otherwise make sure you have a companies_list, start and end date 
summary_df, returns_df = qof.get_assets(companies_list = Target_Tickers, start_date=START_DATE, end_date=END_DATE)

Asset_Names = returns_df.columns
Number_of_Assets = returns_df.shape[1] 

# Asset_Returns_DataFrame = Asset_Returns_DataFrame.reset_index(drop=True)

if print_flag: 
    print(returns_df.head())

                AAPL       IBM       JNJ       CVS
Date                                              
2020-01-03 -0.009722 -0.007975 -0.011578 -0.007956
2020-01-06  0.007968 -0.001787 -0.001248  0.003942
2020-01-07 -0.004703  0.000671  0.006107 -0.003791
2020-01-08  0.016086  0.008346 -0.000138 -0.012503
2020-01-09  0.021241  0.010568  0.002966  0.002753


### Compute Expected Returns and Mean-Variance Matrix 

In [5]:
Expected_Return_Vector = compute_expected_return_vector(returns_df)

Sigma_Covariance_Matrix = compute_covariance_matrix(returns_df)

In [6]:
# Display matricies 

if print_flag: 
    print("Expected Returns")
    print(Expected_Return_Vector)

    print("\nCovariance Matrix")
    print(Sigma_Covariance_Matrix)

Expected Returns
[ 1.15745738e-03  5.65255162e-04  6.94732284e-05 -2.12780453e-04]

Covariance Matrix
[[3.98286266e-04 1.34429353e-04 9.28265508e-05 1.13966117e-04]
 [1.34429353e-04 2.86334050e-04 1.00528109e-04 1.29964490e-04]
 [9.28265508e-05 1.00528109e-04 1.54291276e-04 1.01518272e-04]
 [1.13966117e-04 1.29964490e-04 1.01518272e-04 3.68772307e-04]]


### Build QUBO Matrix 

In [7]:
Q_Matrix = build_qubo_matrix_mean_variance(
    Sigma_Covariance_Matrix=Sigma_Covariance_Matrix,
    Expected_Return_Vector=Expected_Return_Vector,
    Risk_Aversion_Parameter=RISK_AVERSION
)

In [8]:
if print_flag: 
    print(pd.DataFrame(Q_Matrix, index=Asset_Names, columns=Asset_Names))

          AAPL       IBM       JNJ       CVS
AAPL -0.000958  0.000067  0.000046  0.000057
IBM   0.000067 -0.000422  0.000050  0.000065
JNJ   0.000046  0.000050  0.000008  0.000051
CVS   0.000057  0.000065  0.000051  0.000397


## QAOA 